# Episode 1 — What is PYNQ?

**PYNQ** (Python Productivity for Zynq) lets you control FPGA hardware from Python, in a web browser.

The Zynq chip on the EDGE ZYNQ board has two halves:

| Half | What it is | On this board |
| --- | --- | --- |
| **PS** – Processing System | Two ARM Cortex-A9 cores running Ubuntu Linux, Python and Jupyter | 512 MB DDR3, Ethernet, USB, microSD |
| **PL** – Programmable Logic | FPGA fabric that holds *your* hardware | LEDs, switches, buttons, 7-segment, buzzer, XADC, HDMI |

The two halves talk over **AXI** buses: every hardware block in the PL appears to Python as a range of memory addresses.

An **overlay** is a hardware design for the PL. It is two files:
* `name.bit` – the bitstream that configures the FPGA
* `name.hwh` – a description of the blocks inside it, their addresses and interrupts, which PYNQ reads to build Python objects for you

In this episode you will load the EDGE **base overlay**, look inside it, and control the board from Python.

Runs on: EDGE ZYNQ Z7-10 and Z7-20.

## 1. Which board is this?

In [ ]:
import pynq
from pynq.overlays.base import board_variant

print("PYNQ version:", pynq.__version__)
print("Board:", board_variant())

## 2. Load the base overlay

Loading an overlay programs the PL. It takes about a second.

In [ ]:
from pynq.overlays.base import BaseOverlay

base = BaseOverlay("base.bit")
print("Overlay loaded:", base.is_loaded())

## 3. Look inside the overlay

PYNQ read `base.hwh` and found every hardware block, its address and the Python driver it will use.

In [ ]:
for name, ip in base.ip_dict.items():
    if "phys_addr" not in ip:          # the processor itself has no address
        continue
    print("{:42s} 0x{:08X}  {}".format(name, ip["phys_addr"], ip["type"]))

## 4. Control the hardware

These are ordinary Python objects. Each call writes to the block's address in the PL.

In [ ]:
import time

base.leds[0].on()
time.sleep(1)
base.leds[0].off()

base.seven_seg.show_text("PYNQ")
print("Switches:", format(base.switches.read(), "04b"))
print("Chip temperature: {:.1f} °C".format(base.xadc.temperature()))

## 5. Under the hood: it is just memory

A driver is a thin layer over memory-mapped I/O (MMIO). Here we skip the driver and write
straight to the LED block's data register at offset 0.

In [ ]:
from pynq import MMIO

led_addr = base.ip_dict["leds_gpio"]["phys_addr"]
leds = MMIO(led_addr, 0x10000)

for value in [0b0001, 0b0011, 0b0111, 0b1111, 0b0000]:
    leds.write(0x0, value)        # register 0 = the LED outputs
    time.sleep(0.3)

## 6. Hardware keeps running without Python

The 7-segment display is multiplexed by logic in the PL: Python wrote one register,
and the hardware refreshes the four digits on its own, about 380 times a second.

In [ ]:
base.seven_seg.show_number(2026)

## Try it yourself

Make the four LEDs count from 0 to 15 in binary, one step every half second.
Hint: `base.leds[0:4].write(n)` writes all four at once.

In [ ]:
# your code here